# ResNet50V2 Skin-Lesion Classifier — Colab Version + Hyperparameter Comparison

This is the Colab version of the hyperparameter-comparison notebook (same as before, just with Google Drive
paths instead of Kaggle input paths — for whenever you'd rather stay on Colab).

**What was already fixed in your original notebook (so these are NOT re-explored below unless you want to):**
- Optimizer: Adam
- Learning rate: 0.001
- Batch size: 64
- Epochs: 70
- Base model: ResNet50V2, fully frozen (no fine-tuning)
- Head: GAP → Dense(256) → Dense(128) → Dense(64) → Dense(12), no Dropout
- No LR schedule (imported `ReduceLROnPlateau`/`LearningRateScheduler` but never used them)
- Only callback used: `ModelCheckpoint`

**What this notebook lets you vary and compare:**
1. Learning rate
2. Batch size
3. Optimizer (Adam / SGD+momentum / RMSprop / AdamW)
4. Fine-tuning depth (how many top ResNet50V2 layers you unfreeze, instead of freezing all)
5. Dropout in the classification head
6. LR scheduling (`ReduceLROnPlateau`) + `EarlyStopping` (saves GPU time)

See **"How to run this on Colab"** at the very bottom before you start.

In [ ]:
# --- GPU check ---
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs found:", gpus)
if not gpus:
    print("WARNING: No GPU detected. On Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4).")


In [ ]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- Imports ---
import os, time, pathlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, AdamW
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                              jaccard_score, cohen_kappa_score, matthews_corrcoef)

# Optional: mixed precision. On a Colab T4 this roughly halves memory use and speeds up
# training, which matters when your GPU time is limited. Safe to leave on for ResNet.
USE_MIXED_PRECISION = True
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled:", tf.keras.mixed_precision.global_policy())


## 1. Data paths (Google Drive)

Same layout you already used: your `split_dataset` folder (with `train/`, `val/`, `test/` subfolders, each
containing one subfolder per class) living somewhere in your Drive.

In [ ]:
train_data = "/content/drive/MyDrive/defense/split_dataset/train"
valid_data = "/content/drive/MyDrive/defense/split_dataset/val"
test_data  = "/content/drive/MyDrive/defense/split_dataset/test"

dataset_dir1, dataset_dir2, dataset_dir3 = map(pathlib.Path, [train_data, valid_data, test_data])

# Where checkpoints/plots/results get saved (on Drive, so they survive when the Colab runtime resets)
OUTPUT_DIR = "/content/drive/MyDrive/defense/save"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Training images:', len(list(dataset_dir1.glob('*/*.jpg'))))
print('Validation images:', len(list(dataset_dir2.glob('*/*.jpg'))))
print('Testing images:', len(list(dataset_dir3.glob('*/*.jpg'))))
print('Classes:', sorted(os.listdir(train_data)))


In [ ]:
IMAGE_SIZE = (224, 224)
NUM_CLASSES = len(os.listdir(train_data))
print("Number of classes:", NUM_CLASSES)


## 2. Reusable builders

Everything that used to be hard-coded (batch size baked into the generator call, learning rate baked into
`Adam(learning_rate=0.001)`, optimizer baked in as `Adam`, base model always fully frozen) is now a function
argument, so the same code can be reused for every experiment.

In [ ]:
def build_data_generators(batch_size, augment=True):
    if augment:
        train_datagen = ImageDataGenerator(
            rescale=1.0 / 255,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )
    else:
        train_datagen = ImageDataGenerator(rescale=1.0 / 255)

    valid_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_generator = train_datagen.flow_from_directory(
        train_data, target_size=IMAGE_SIZE, batch_size=batch_size,
        class_mode='categorical'
    )
    valid_generator = valid_datagen.flow_from_directory(
        valid_data, target_size=IMAGE_SIZE, batch_size=batch_size,
        class_mode='categorical'
    )
    return train_generator, valid_generator


def get_optimizer(name, learning_rate):
    name = name.lower()
    if name == 'adam':
        return Adam(learning_rate=learning_rate)
    if name == 'adamw':
        return AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    if name == 'sgd':
        return SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    if name == 'rmsprop':
        return RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def build_model(learning_rate=0.001, optimizer_name='adam', dropout_rate=0.0,
                 fine_tune_at=None, dense_units=(256, 128, 64)):
    """
    fine_tune_at:
        None  -> freeze the entire ResNet50V2 base (matches your original notebook)
        int N -> unfreeze only the last N layers of the base model, freeze the rest
                 (lets the backbone adapt a little to your data instead of only the head)
    """
    base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))

    if fine_tune_at is None:
        base_model.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers[:-fine_tune_at]:
            layer.trainable = False

    input_tensor = Input(shape=(*IMAGE_SIZE, 3))
    x = base_model(input_tensor)
    x = GlobalAveragePooling2D()(x)
    for units in dense_units:
        x = Dense(units, activation='relu')(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate)(x)
    # Force float32 output for numerical stability under mixed precision
    output = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = Model(inputs=input_tensor, outputs=output)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=get_optimizer(optimizer_name, learning_rate),
        metrics=['accuracy']
    )
    return model


## 3. Hyperparameter search (short runs, to compare cheaply)

Instead of running every combination for the full 70 epochs (which would burn through your limited GPU time
fast), each config below trains for only `EPOCHS_SEARCH` epochs with `EarlyStopping`, so a config that isn't
working stops early. We record the **best validation accuracy/loss** and **wall-clock time** for each, then
compare.

Kept deliberately small (8 configs) so it's realistic on a single Colab GPU session — edit `EXPERIMENTS` to
add or remove configs. Each config changes **one or two things at a time** so you can read off what mattered.
Note: Colab free-tier sessions can disconnect after ~90 min idle or ~12 hrs total, so results and checkpoints
here are saved to Drive (`OUTPUT_DIR`) as you go, not just kept in memory.

In [ ]:
EPOCHS_SEARCH = 10          # short budget per experiment; raise a bit if you have quota to spare
SEARCH_PATIENCE = 3        # EarlyStopping patience during the search phase

# name -> dict of kwargs for build_data_generators / build_model
EXPERIMENTS = [
    {"name": "baseline (lr=1e-3, bs=64, adam, frozen)",        "lr": 1e-3,  "batch_size": 64,  "optimizer": "adam",    "dropout": 0.0, "fine_tune_at": None},
    {"name": "lower lr (lr=1e-4)",                              "lr": 1e-4,  "batch_size": 64,  "optimizer": "adam",    "dropout": 0.0, "fine_tune_at": None},
    {"name": "higher lr (lr=1e-2)",                             "lr": 1e-2,  "batch_size": 64,  "optimizer": "adam",    "dropout": 0.0, "fine_tune_at": None},
    {"name": "smaller batch (bs=32)",                           "lr": 1e-3,  "batch_size": 32,  "optimizer": "adam",    "dropout": 0.0, "fine_tune_at": None},
    {"name": "larger batch (bs=128)",                           "lr": 1e-3,  "batch_size": 128, "optimizer": "adam",    "dropout": 0.0, "fine_tune_at": None},
    {"name": "SGD+momentum (lr=1e-2)",                          "lr": 1e-2,  "batch_size": 64,  "optimizer": "sgd",     "dropout": 0.0, "fine_tune_at": None},
    {"name": "RMSprop (lr=1e-3)",                                "lr": 1e-3,  "batch_size": 64,  "optimizer": "rmsprop", "dropout": 0.0, "fine_tune_at": None},
    {"name": "dropout 0.3 + fine-tune last 20 layers",           "lr": 1e-4,  "batch_size": 64,  "optimizer": "adam",    "dropout": 0.3, "fine_tune_at": 20},
]

def run_experiment(cfg):
    tf.keras.backend.clear_session()
    train_gen, valid_gen = build_data_generators(cfg["batch_size"])
    model = build_model(
        learning_rate=cfg["lr"],
        optimizer_name=cfg["optimizer"],
        dropout_rate=cfg["dropout"],
        fine_tune_at=cfg["fine_tune_at"],
    )

    early_stop = EarlyStopping(monitor='val_accuracy', patience=SEARCH_PATIENCE,
                                restore_best_weights=True, mode='max')

    t0 = time.time()
    hist = model.fit(
        train_gen,
        steps_per_epoch=train_gen.samples // train_gen.batch_size,
        epochs=EPOCHS_SEARCH,
        validation_data=valid_gen,
        validation_steps=valid_gen.samples // valid_gen.batch_size,
        callbacks=[early_stop],
        verbose=1,
    )
    elapsed = time.time() - t0

    best_epoch = int(np.argmax(hist.history['val_accuracy']))
    return {
        "name": cfg["name"],
        "lr": cfg["lr"],
        "batch_size": cfg["batch_size"],
        "optimizer": cfg["optimizer"],
        "dropout": cfg["dropout"],
        "fine_tune_at": cfg["fine_tune_at"],
        "best_val_accuracy": hist.history['val_accuracy'][best_epoch],
        "best_val_loss": hist.history['val_loss'][best_epoch],
        "epochs_run": len(hist.history['val_accuracy']),
        "seconds": round(elapsed, 1),
    }

results = []
for cfg in EXPERIMENTS:
    print(f"\n=== Running: {cfg['name']} ===")
    results.append(run_experiment(cfg))
    # Save after each experiment so a Colab disconnect doesn't lose earlier results
    pd.DataFrame(results).to_csv(os.path.join(OUTPUT_DIR, "hparam_search_results.csv"), index=False)

results_df = pd.DataFrame(results).sort_values("best_val_accuracy", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Visual comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="best_val_accuracy", y="name", palette="viridis")
plt.xlabel("Best validation accuracy (short run)")
plt.ylabel("")
plt.title("Hyperparameter search — validation accuracy comparison")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hparam_search_comparison.png"))
plt.show()


## 4. Final full training with the best config

Pick the top row of `results_df` (or a config you liked for other reasons, e.g. fewer seconds/epoch if two rows
are close) and set the variables below. This cell then trains for the full `EPOCHS_FULL` epochs like your
original notebook, but now with `ReduceLROnPlateau` and `EarlyStopping` added — these didn't do anything in your
original notebook (the imports were unused), and they matter most here: they stop a bad run early and shrink the
learning rate automatically when validation performance plateaus, saving GPU hours.

In [ ]:
# --- Set these from the best row of results_df (or your own choice) ---
best = results_df.iloc[0]
BEST_LR          = float(best["lr"])
BEST_BATCH_SIZE  = int(best["batch_size"])
BEST_OPTIMIZER   = best["optimizer"]
BEST_DROPOUT     = float(best["dropout"])
BEST_FINE_TUNE_AT = None if pd.isna(best["fine_tune_at"]) else int(best["fine_tune_at"])

EPOCHS_FULL = 70  # same as your original notebook

print("Training final model with:", dict(
    lr=BEST_LR, batch_size=BEST_BATCH_SIZE, optimizer=BEST_OPTIMIZER,
    dropout=BEST_DROPOUT, fine_tune_at=BEST_FINE_TUNE_AT
))

tf.keras.backend.clear_session()
train_generator, valid_generator = build_data_generators(BEST_BATCH_SIZE)
model = build_model(
    learning_rate=BEST_LR,
    optimizer_name=BEST_OPTIMIZER,
    dropout_rate=BEST_DROPOUT,
    fine_tune_at=BEST_FINE_TUNE_AT,
)
model.summary()


In [ ]:
checkpoint_path = os.path.join(OUTPUT_DIR, "Finalversion.weights.h5")

model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    save_best_only=True,
    monitor='val_accuracy',
    save_freq='epoch',
    save_weights_only=True
)
early_stop = EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)

start_time = time.time()
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=EPOCHS_FULL,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // valid_generator.batch_size,
    callbacks=[model_checkpoint, early_stop, reduce_lr],
)
training_duration = time.time() - start_time
print(f"Training completed in: {training_duration:.1f} seconds")


In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend(); plt.title("Loss"); plt.savefig(os.path.join(OUTPUT_DIR, 'resnet_LossVal_loss.png')); plt.show()

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend(); plt.title("Accuracy"); plt.savefig(os.path.join(OUTPUT_DIR, 'resnet_AccVal_acc.png')); plt.show()


In [ ]:
train_loss, train_acc = model.evaluate(train_generator)
print("Train accuracy:", train_acc)
valid_loss, valid_acc = model.evaluate(valid_generator)
print("Validation accuracy:", valid_acc)


## 5. Test-set evaluation (same metrics as your original notebook)

In [ ]:
test_datagen_obj = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen_obj.flow_from_directory(
    test_data, target_size=IMAGE_SIZE, batch_size=BEST_BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_loss, test_acc = model.evaluate(test_generator)
print("Test accuracy:", test_acc)


In [ ]:
true_labels = test_generator.classes
pred_probs = model.predict(test_generator)
predicted_labels = np.argmax(pred_probs, axis=1)

confusion = confusion_matrix(true_labels, predicted_labels)
class_labels = list(test_generator.class_indices.keys())

plt.figure(figsize=(8, 6))
sns.set(font_scale=1.0)
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Labels"); plt.ylabel("True Labels"); plt.title("Confusion Matrix")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png')); plt.show()

report = classification_report(true_labels, predicted_labels, target_names=class_labels, output_dict=True)
accuracy = accuracy_score(true_labels, predicted_labels)

for class_label in class_labels:
    print(f"Class: {class_label}")
    print(f"  Precision: {report[class_label]['precision']:.2f}")
    print(f"  Recall: {report[class_label]['recall']:.2f}")
    print(f"  F1-score: {report[class_label]['f1-score']:.2f}")
    idx = class_labels.index(class_label)
    print(f"  Class Accuracy: {confusion[idx, idx] / np.sum(confusion[idx, :]):.2%}")
    print(f"  Support: {report[class_label]['support']}\n")

print(f"Overall Precision: {report['macro avg']['precision']:.2f}")
print(f"Overall Recall: {report['macro avg']['recall']:.2f}")
print(f"Overall F1-score: {report['macro avg']['f1-score']:.2f}")
print(f"Overall Accuracy: {accuracy:.2%}")


In [ ]:
misclassification_error = 1 - accuracy_score(true_labels, predicted_labels)
jaccard_similarity = jaccard_score(true_labels, predicted_labels, average='micro')
kappa = cohen_kappa_score(true_labels, predicted_labels)
mcc = matthews_corrcoef(true_labels, predicted_labels)

print("Misclassification Error:", misclassification_error)
print("Jaccard Similarity Score:", jaccard_similarity)
print("Cohen's Kappa:", kappa)
print("Matthews Correlation Coefficient (MCC):", mcc)


In [ ]:
# ============================================================
# DOWNLOAD ALL GENERATED PLOTS/IMAGES FROM THIS COLAB RUNTIME
# ============================================================

import os
import glob
import zipfile
from google.colab import drive, files

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Folders where your notebook saves images
search_folders = [
    "/content/drive/MyDrive/defense/save",
    "/content/drive/MyDrive/defense/save/cross_validation",
    "/content"
]

# 3. Find all image files
image_files = []

for folder in search_folders:
    if os.path.exists(folder):
        for root, dirs, files_list in os.walk(folder):
            for filename in files_list:
                if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                    full_path = os.path.join(root, filename)

                    # Avoid duplicate files
                    if full_path not in image_files:
                        image_files.append(full_path)

# 4. Remove possible duplicates caused by searching parent folders
image_files = sorted(set(image_files))

print("=" * 60)
print("IMAGES FOUND")
print("=" * 60)

if not image_files:
    print("❌ No images were found.")
    print()
    print("This means the plotting cells probably have not been executed yet.")
else:
    for i, path in enumerate(image_files, 1):
        print(f"{i}. {path}")

    print()
    print(f"Total images found: {len(image_files)}")

    # 5. Create ZIP file
    zip_path = "/content/all_generated_images.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for path in image_files:
            # Keep only filename in ZIP to avoid huge folder structures
            zipf.write(path, arcname=os.path.basename(path))

    print()
    print("=" * 60)
    print("ZIP CREATED")
    print("=" * 60)
    print(zip_path)

    # 6. Download ZIP
    files.download(zip_path)hyuj7g  kiyujtrfgvaz uy7